<a href="https://colab.research.google.com/github/BrianAyekha/-connection-of-MySQL-database-to-NodeJs/blob/main/Week3_Descriptive_Statistics_Correlation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week4_Exploring_Mathematics_of_Data.ipynb
**Project:** Early fault detection and diagnosis in hospital electromechanical equipment (infant warmer case study)

**Dataset:** AI4I 2020 Predictive Maintenance Dataset (UCI, id 601), the same dataset imported in the previous notebook.

**Author:** Brian Ayekha Mwangala

## 1. Load the dataset

In [ ]:
!pip install -q ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

# Download the dataset and join inputs (features) and targets into one table
ai4i = fetch_ucirepo(id=601)
df = pd.concat([ai4i.data.features, ai4i.data.targets], axis=1)

df.head()

## 2. Select numerical variables
I chose four numerical variables that map to quantities in an infant warmer:
- `Air temperature` (K): like ambient temperature
- `Process temperature` (K): like heater or mattress temperature
- `Rotational speed` (rpm): like load on the system
- `Torque` (Nm): like load on the system (e.g. supply current)

In [ ]:
variables = ["Air temperature", "Process temperature", "Rotational speed", "Torque"]
data = df[variables]
data.head()

## 3. Mean, median, range, variance and standard deviation

In [ ]:
# Calculate each statistic for every selected variable
# Note: pandas uses the sample variance/std (divides by n - 1)
stats = pd.DataFrame({
    "Mean":     data.mean(),
    "Median":   data.median(),
    "Min":      data.min(),
    "Max":      data.max(),
    "Range":    data.max() - data.min(),     # range = max - min
    "Variance": data.var(),
    "Std Dev":  data.std(),
})

stats.round(2)

## 4. Covariance and correlation between two variables

In [ ]:
# Chosen pair: Air temperature (x) and Process temperature (y)
x = df["Air temperature"]
y = df["Process temperature"]

# Covariance: how the two variables vary together (depends on the units)
cov_xy = np.cov(x, y)[0, 1]

# Pearson correlation: covariance scaled to between -1 and +1 (no units)
corr_xy = np.corrcoef(x, y)[0, 1]

print(f"Covariance : {cov_xy:.4f}  (K^2)")
print(f"Correlation: {corr_xy:.4f}")

## 5. Plot: Air temperature vs Process temperature

In [ ]:
# Fit a straight line y = w*x + b to show the trend (used again in Section 7)
w, b = np.polyfit(x, y, 1)

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.2, s=10, label="Data points")   # alpha makes overlapping points visible
plt.plot(np.sort(x), w * np.sort(x) + b, color="red", label=f"Fitted line: y = {w:.2f}x + {b:.1f}")
plt.xlabel("Air temperature (K)")
plt.ylabel("Process temperature (K)")
plt.title(f"Air vs Process temperature (r = {corr_xy:.2f})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Interpretation

**What do the statistics tell me about the data?**
Compare the table above with these points. The temperatures have a small standard deviation compared with their means, so air and process temperature stay in a narrow band, which is typical of a controlled process. Rotational speed and torque spread much more. When the mean and median of a variable are close, its distribution is roughly symmetric. When the mean is clearly larger than the median (I expect this for rotational speed), a few high values pull it up and the distribution is skewed. The variables also have very different units and scales (K, rpm, Nm), so they will need scaling before many ML models. Variance and standard deviation tell me how much each variable fluctuates, which is the "normal behaviour" an anomaly detector would learn.

**What does the correlation tell me?**
The correlation coefficient `r` (printed in Section 4) measures the strength and direction of the *linear* relationship between air and process temperature. A value near +1 means that when air temperature is higher, process temperature tends to be higher too; near −1 means they move in opposite directions; near 0 means no linear relationship. The sign and size of `r` from my output tell me how tightly the points follow the fitted line. The covariance has the same sign as `r`, but its size depends on the units, so it is hard to compare across variable pairs.

**Does correlation mean causation?**
No. A strong correlation only shows that two variables move together. It does not show that one causes the other. A third factor could drive both, or the relationship could be a coincidence. Here a physical link is plausible, since heat from the surroundings affects the process temperature, but the correlation alone does not prove it. In this dataset the data is synthetic, and I would need domain knowledge or a controlled experiment, such as changing the ambient temperature on my warmer prototype and measuring the response, to claim cause and effect.

## 7. Input → Function/Model → Output

I choose **Air temperature as the input** ($x$) and **Process temperature as the output** ($y$):

$$x \;\longrightarrow\; \boxed{\,f(x)\,} \;\longrightarrow\; y$$

A simple model is a straight line, with parameters $w$ (slope) and $b$ (intercept) that are learned from the data:

$$y = f(x) = w\,x + b$$

Here: $x$ = air temperature (K), $y$ = predicted process temperature (K).

The main goal of my project follows the same idea with several inputs and a class as the output:

$$\text{Machine failure} = f(\text{air temp},\ \text{process temp},\ \text{speed},\ \text{torque},\ \text{tool wear})$$

In [ ]:
# The same idea in Python: input -> model -> output
def model(air_temp, w=w, b=b):
    # f(x) = w*x + b, with w and b learned from the data in Section 5
    return w * air_temp + b

input_value = 300.0                      # input: air temperature in K
output_value = model(input_value)        # output: predicted process temperature
print(f"Input (air temp) = {input_value} K  ->  Output (process temp) = {output_value:.2f} K")